# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR^2 dataset using the `mlcroissant` library. We'll walk step-by-step through loading the Croissant-based dataset, reviewing its metadata and schema, extracting record sets using their `@id`, and conducting basic analysis.

### Dataset Source
The dataset is described by a Croissant schema:
- URL: https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Install mlcroissant if necessary
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the full Croissant metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {getattr(dataset.metadata, 'name', 'N/A')}")
print(f"Description: {getattr(dataset.metadata, 'description', 'N/A')}")

## 2. Data Overview

Let's enumerate all available **Record Sets**, their fields, and columns, referencing everything explicitly by their `@id`.

We print record set `@id`s and then for each, fields and columns present in the schema.

In [ ]:
print("Available Record Sets (by @id):")
record_sets = []
for rs in getattr(dataset.metadata, 'record_set', []):
    rs_id = getattr(rs, '@id', None)
    if rs_id:
        record_sets.append(rs_id)
        print(f"- {rs_id}")

# For each record set, list its fields and columns
for rs in getattr(dataset.metadata, 'record_set', []):
    rs_id = getattr(rs, '@id', None)
    print(f"\nRecord Set @id: {rs_id}")
    print(" Fields:")
    for field in getattr(rs, 'field', []):
        print(f"  - {getattr(field, '@id', '')} (name: {getattr(field, 'name', '')})")
    print(" Columns:")
    for column in getattr(rs, 'column', []):
        print(f"  - {getattr(column, '@id', '')} (name: {getattr(column, 'name', '')})")

## 3. Data Extraction

Let's extract all data available as records for each record set, referencing the record set by its `@id`. Data is loaded into Pandas DataFrames for further analysis.

In [ ]:
# Build mapping from @id to RecordSet object for clarity
record_set_objs = {getattr(rs, '@id'): rs for rs in getattr(dataset.metadata, 'record_set', [])}

dataframes = {}

# You may adjust this list if you wish to extract only a subset
ids_to_use = record_sets  # All found above

for record_set_id in ids_to_use:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  - Loaded {len(df)} records; columns: {df.columns.tolist()}")
    else:
        print(f"  - No records found.")

# Show the first five rows of the first record set if available
if dataframes:
    first_set = ids_to_use[0]
    display(dataframes[first_set].head())

## 4. Exploratory Data Analysis (EDA)

Let's select a record set with numeric fields and demonstrate filtering, normalization, and grouping, referencing everything by `@id`.

In [ ]:
import numpy as np

# We'll proceed with the first record set and try to auto-detect a numeric field
if dataframes:
    record_set_id = ids_to_use[0]
    df = dataframes[record_set_id]
    print(f"Analyzing Record Set: {record_set_id}")
    # Try to find a numeric column (float or int)
    numeric_fields = []
    for col in df.columns:
        try:
            # See if at least half the column can be converted to float
            vals = pd.to_numeric(df[col], errors='coerce')
            if vals.notna().mean() > 0.5:
                numeric_fields.append(col)
        except Exception:
            continue
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field (by @id/column name): {numeric_field}")

        # Clean/convert the numeric field
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanmean(df[numeric_field])
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > mean ({threshold:.2f}): {len(filtered_df)} records")

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try to group by a string-type field
        possible_group_fields = [col for col in df.columns if df[col].dtype==object and col != numeric_field]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            print(f"Grouping by: {group_field}")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped_df.head())
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No dataframes found to analyze.")

## 5. Visualization

Let's visualize the distribution of one numeric field. We'll plot a histogram and, if grouped values are available, we'll also plot group-wise means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram for the numeric field in main record set
if dataframes:
    record_set_id = ids_to_use[0]
    df = dataframes[record_set_id]
    # Recompute numeric fields
    numeric_fields = []
    for col in df.columns:
        try:
            vals = pd.to_numeric(df[col], errors='coerce')
            if vals.notna().mean() > 0.5:
                numeric_fields.append(col)
        except Exception:
            continue
    if numeric_fields:
        numeric_field = numeric_fields[0]
        fig, ax = plt.subplots(figsize=(7,4))
        sns.histplot(pd.to_numeric(df[numeric_field], errors='coerce').dropna(), bins=30, ax=ax, kde=True)
        ax.set_title(f'Distribution of {numeric_field}')
        ax.set_xlabel(numeric_field)
        plt.show()

        # If a group field was found above, plot group means
        possible_group_fields = [col for col in df.columns if df[col].dtype==object and col != numeric_field]
        if possible_group_fields:
            group_field = possible_group_fields[0]
            grouped_df = df.groupby(group_field)[numeric_field].mean().reset_index().dropna()
            plt.figure(figsize=(8,4))
            sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
            plt.xticks(rotation=45, ha='right')
            plt.title(f'Mean {numeric_field} by {group_field}')
            plt.tight_layout()
            plt.show()
else:
    print("No dataframes to visualize.")

## 6. Conclusion

- We loaded the FAIR^2 dataset using `mlcroissant` and explored its Croissant schema via `@id` identifiers.
- We extracted and loaded available record sets into DataFrames for analysis.
- We demonstrated basic filtering, normalization, grouping, and visualization for numerical and categorical fields.

This workflow highlights how Croissant-encoded datasets can be explored programmatically and interoperably using only schema references. For more in-depth analysis, consult the schema documentation to identify additional record sets and field relationships.